# QLoRA Fine-Tuning: TinyLlama on Databricks Dolly

Fine-tuning TinyLlama-1.1B with QLoRA on the Dolly instruction dataset, then checking whether the fine-tuned model actually generates better instruction-following responses than the base model on unseen examples.

**Workflow:** load and split the Dolly dataset -> format as instruction/response pairs -> tokenize -> generate baseline responses from the plain model -> load TinyLlama in 4-bit (NF4) -> attach a LoRA adapter -> train -> evaluate with held-out loss and perplexity -> compare base vs fine-tuned generations on the same prompts -> merge and save the adapter.

**Stack:** PyTorch, Hugging Face Transformers, PEFT, bitsandbytes, Datasets.

Runs end-to-end on a single Colab T4.

## Setup

In [1]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 15.4 MB/s eta 0:00:00


## Imports and Configuration

In [2]:
import math
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset

assert torch.cuda.is_available(), "QLoRA needs a CUDA GPU. Switch to a GPU runtime."

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DATASET_NAME = "databricks/databricks-dolly-15k"
NUM_EXAMPLES = 2000
EVAL_SIZE = 200
MAX_SEQ_LENGTH = 256
N_COMPARISON_EXAMPLES = 8

OUTPUT_DIR = "./qlora-tinyllama-dolly"
MERGED_MODEL_DIR = "./qlora-tinyllama-dolly-merged"

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

TRAIN_BATCH_SIZE = 4
GRAD_ACCUM_STEPS = 4
NUM_EPOCHS = 2
LEARNING_RATE = 2e-4

## Dataset

About 2,000 examples from Dolly, split into ~1,800 for training and 200 held out for evaluation. The held-out split is never used for gradient updates.

In [3]:
dataset = load_dataset(DATASET_NAME, split="train")
dataset = dataset.shuffle(seed=SEED).select(range(NUM_EXAMPLES))

split_dataset = dataset.train_test_split(test_size=EVAL_SIZE, seed=SEED)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"Total examples: {len(dataset)}")
print(f"Training examples: {len(train_dataset)}")
print(f"Evaluation examples: {len(eval_dataset)}")
print()
print(train_dataset[0])
print()
print(train_dataset[1])

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl: reconstructing file:   0%|          |  0.00B / 13.1MB            

databricks-dolly-15k.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Total examples: 2000
Training examples: 1800
Evaluation examples: 200

{'instruction': 'Think of a number of different ways you can cook eggs', 'context': '', 'response': 'Eggs are versatile and can be cooked using a number of methods including boiling, scrambling, poaching, frying and baking. They can also be beaten and augmented with other ingredients to make an Omelette or Frittata', 'category': 'brainstorming'}

{'instruction': 'What is the state bird of Texas?', 'context': '', 'response': 'Mockingbird', 'category': 'open_qa'}


## Data Formatting

In [4]:
def format_example(example):
    instruction = example["instruction"]
    context = example["context"]
    response = example["response"]

    if context and context.strip():
        text = (
            f"Instruction: {instruction}\n"
            f"Context: {context}\n"
            f"Response: {response}"
        )
    else:
        text = (
            f"Instruction: {instruction}\n"
            f"Response: {response}"
        )

    example["text"] = text
    return example


def build_prompt(example):
    instruction = example["instruction"]
    context = example["context"]

    if context and context.strip():
        return f"Instruction: {instruction}\nContext: {context}\nResponse:"
    return f"Instruction: {instruction}\nResponse:"


formatted_train_dataset = train_dataset.map(format_example)
formatted_eval_dataset = eval_dataset.map(format_example)

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

## Tokenization

Labels are set to the full tokenized sequence, so training uses a standard causal LM objective over instruction + context + response rather than response-only loss. That's a simplification, not response-masked loss, but it's a reasonable choice for a small dataset and keeps the tokenization step easy to follow. Sequence length is capped at 256 tokens, which covers almost all Dolly examples without pushing T4 memory.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens


tokenized_train_dataset = formatted_train_dataset.map(
    tokenize, batched=False, remove_columns=formatted_train_dataset.column_names
)
tokenized_eval_dataset = formatted_eval_dataset.map(
    tokenize, batched=False, remove_columns=formatted_eval_dataset.column_names
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

## Baseline Evaluation

Before quantization or LoRA enter the picture, generate from the plain base model on a fixed set of unseen evaluation instructions. The same prompts and generation settings are reused after fine-tuning for a direct comparison.

In [6]:
comparison_examples = eval_dataset.shuffle(seed=SEED).select(range(N_COMPARISON_EXAMPLES))
comparison_instructions = [ex["instruction"] for ex in comparison_examples]
comparison_prompts = [build_prompt(ex) for ex in comparison_examples]


def generate_response(model, tokenizer, prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [7]:
base_model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
)
base_model_fp16.eval()

baseline_responses = [
    generate_response(base_model_fp16, tokenizer, prompt) for prompt in comparison_prompts
]

for instruction, response in zip(comparison_instructions, baseline_responses):
    print(f"Instruction: {instruction}")
    print(f"Base response: {response}")
    print()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Instruction: Who is the psychologist Jonathan Young
Base response: "The Mythic Imagination" in The Journal of the American Society for Psychotherapy and Psychosomatics
"The Hero's Journey in Screenwriting" in The Screenwriter's Handbook
"The Hero's Journey in Screenwriting" in The Screenwriter's Handbook
"The Hero's Journey in Screenwriting" in The Screenwriter's Handbook
"The Hero's Journey in Screen

Instruction: Are UGGs considered fashionable?
Base response: UGGs are considered fashionable. They are comfortable, warm, and stylish. They are perfect for cold weather and are a great option for those who want to look stylish while staying warm.

Instruction: Name some  active NBA famous players
Base response: LeBron James, Stephen Curry, Kevin Durant, Kobe Bryant, and Kareem Abdul-Jabbar are some of the most famous active NBA players.

Instruction: Give me a list of the last five european golden boots winner And tell me how many goals they scored.
Base response: The last five European 

In [8]:
del base_model_fp16
torch.cuda.empty_cache()

## 4-bit Quantization

NF4 with double quantization keeps the base weights in 4-bit while training happens in a higher-precision compute dtype. Compute dtype is set to `float16` rather than `bfloat16` — the T4 (Turing) doesn't have native bf16 tensor core support, so bf16 either falls back to slow emulation or errors out depending on the library version. fp16 is the correct choice for this GPU.

In [9]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## LoRA Configuration

Rank-8 adapters on the attention projections only. `prepare_model_for_kbit_training` handles the usual kbit prep (casting norms, enabling gradient checkpointing hooks, etc.) before the adapter is attached.

In [10]:
base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

qlora_model = get_peft_model(base_model, lora_config)
qlora_model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


## Training

Gradient accumulation keeps the effective batch size reasonable without exceeding T4 memory. `eval_dataset` is passed in for loss tracking only — it's never part of the gradient updates.

In [13]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="no",
    fp16=True,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=qlora_model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,0.934125,1.074956
2,1.007421,1.043125


TrainOutput(global_step=226, training_loss=1.3348999529813244, metrics={'train_runtime': 595.9427, 'train_samples_per_second': 6.041, 'train_steps_per_second': 0.379, 'total_flos': 5726668220006400.0, 'train_loss': 1.3348999529813244, 'epoch': 2.0})

## Evaluation

Loss on the held-out split, converted to perplexity. Both numbers come directly from this run, not hard-coded.

In [14]:
eval_results = trainer.evaluate()
eval_loss = eval_results["eval_loss"]
perplexity = math.exp(eval_loss)

print(f"Evaluation loss: {eval_loss:.4f}")
print(f"Perplexity: {perplexity:.2f}")

Training Loss,Validation Loss,Epoch
1.007421,1.043125,2


Evaluation loss: 1.0431
Perplexity: 2.84


## Model Comparison

Same prompts, same generation settings, now with the fine-tuned model.

In [22]:
qlora_model.eval()

finetuned_responses = [
    generate_response(qlora_model, tokenizer, prompt) for prompt in comparison_prompts
]

for instruction, base_response, ft_response in zip(
    comparison_instructions, baseline_responses, finetuned_responses
):
    print(f"Instruction: {instruction}")
    print(f"Base:        {base_response}")
    print(f"Fine-tuned:  {ft_response}")
    print()

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Instruction: Who is the psychologist Jonathan Young
Base:        "The Mythic Imagination" in The Journal of the American Society for Psychotherapy and Psychosomatics
"The Hero's Journey in Screenwriting" in The Screenwriter's Handbook
"The Hero's Journey in Screenwriting" in The Screenwriter's Handbook
"The Hero's Journey in Screenwriting" in The Screenwriter's Handbook
"The Hero's Journey in Screen
Fine-tuned:  "The Mythic Mind" in The Journal of the American Academy of Religion
"The Hero's Journey in Screenwriting" in Screenwriting: The Art and Craft of Screenwriting
"The Hero's Journey in Screenwriting" in Screenwriting: The Art and Craft of Screenwriting
"The Hero's Journey in Screenwriting" in Screenwriting: The Art and Craft of Screenwriting
"The Hero's Journey

Instruction: Are UGGs considered fashionable?
Base:        UGGs are considered fashionable. They are comfortable, warm, and stylish. They are perfect for cold weather and are a great option for those who want to look styl

## Qualitative Evaluation

Manual side-by-side table for the comparison examples. The score columns are left blank — this is for manual 1-5 scoring, not an automated metric.

## Results

In [21]:
final_train_loss = train_losses[-1] if train_losses else None

print("Training examples:", len(train_dataset))
print("Evaluation examples:", len(eval_dataset))
print("LoRA r / alpha / dropout:", LORA_R, LORA_ALPHA, LORA_DROPOUT)
print("Target modules:", LORA_TARGET_MODULES)
print("Epochs:", NUM_EPOCHS)
print("Final training loss:", final_train_loss)
print("Evaluation loss:", eval_loss)
print("Perplexity:", perplexity)

Training examples: 1800
Evaluation examples: 200
LoRA r / alpha / dropout: 8 16 0.05
Target modules: ['q_proj', 'v_proj']
Epochs: 2
Final training loss: 1.0074212074279785
Evaluation loss: 1.0431251525878906
Perplexity: 2.8380725795144777


## Save Model

Merging folds the LoRA adapter weights into the base model so the result is a single standalone model rather than a base model + adapter pair. The merged model is saved locally for this notebook run only — for anything beyond a local demo, push it to the Hugging Face Hub instead of committing multi-GB weights to a Git repo.

In [20]:
merged_model = qlora_model.merge_and_unload()
merged_model.save_pretrained(MERGED_MODEL_DIR)
tokenizer.save_pretrained(MERGED_MODEL_DIR)

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:377: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./qlora-tinyllama-dolly-merged/tokenizer_config.json',
 './qlora-tinyllama-dolly-merged/chat_template.jinja',
 './qlora-tinyllama-dolly-merged/tokenizer.json')

## Summary

TinyLlama-1.1B-Chat was fine-tuned with a rank-8 QLoRA adapter (NF4, double-quantized) on ~1,800 Dolly instruction examples, holding out 200 examples for evaluation. Only the attention projection adapters were trained; the base weights stayed frozen and quantized.

The evaluation loss and perplexity computed above are the actual numbers from this run — see the Results section rather than this summary for the exact values. The base-vs-fine-tuned comparison table shows whether responses got more on-topic and instruction-shaped after training; the scoring columns are left for manual review since no automated metric was added here.

**Limitations:** 2,000 training examples and 2 epochs is a small, fast run meant for a portfolio project, not a production fine-tune. Labels cover the full sequence rather than response-only tokens, so some of the loss signal comes from the instruction/context text itself. TinyLlama's 1.1B parameter count limits how much instruction-following quality it can reach regardless of fine-tuning. Results should be read as a demonstration of the QLoRA workflow, not a benchmark claim — if the comparison table shows mixed results on some prompts, that's expected at this scale.